# Willingness-to-pay-space mixed logit

## Model

The utility is parameterized directly in monetary trade-offs:

\[
V_{njr}=V^{(0)}_{nj}
+\alpha\left(c_{nj}
+w^{\mathrm{time}}_{nr}t_{nj}
+w^{\mathrm{crowd}}_{nr}q_{nj}\right),
\]

where \(\alpha<0\) is the cost coefficient and

\[
\boldsymbol w_{nr}
=\boldsymbol\mu_w+\boldsymbol L_w\boldsymbol z_r.
\]

The example estimates random willingness to pay for travel time and crowding,
together with alternative constants and the cost coefficient. Each simulated
traveler has eight repeated choices, so the random WTP vector is shared within
traveler.
The two distribution scales are fixed at their data-generating values in this
compact example. Set `sigma_fixed=False` to estimate them in a larger panel.
Because crowding is undesirable, its WTP enters with the same sign convention
as generalized cost.

This notebook is self-contained and was executed in the repository's Office
validation environment. Install the released package with
`pip install torchdcm`, then select CPU or CUDA through the `device` argument.


In [1]:
import numpy as np
import torch

from IPython.display import HTML, display

from torchdcm import Beta, ChoiceDataset, UtilitySpec, WTPCoefficient, WTPMixedLogit
from torchdcm.datasets import make_swissmetro_like
torch.set_default_dtype(torch.float64)
torch.manual_seed(7)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"TorchDCM example running on {device}")


TorchDCM example running on cuda


In [2]:
rng = np.random.default_rng(15)
frame = make_swissmetro_like(n_obs=960, seed=15)
frame["person_id"] = np.arange(len(frame)) // 8
alternatives = np.array(["TRAIN", "SM", "CAR"])
time = frame[["time_train", "time_sm", "time_car"]].to_numpy()
cost = frame[["cost_train", "cost_sm", "cost_car"]].to_numpy()
available = frame[["avail_train", "avail_sm", "avail_car"]].to_numpy(bool)
crowding = np.column_stack(
    [
        rng.uniform(4.0, 9.0, len(frame)),
        rng.uniform(2.0, 7.0, len(frame)),
        rng.uniform(0.0, 1.0, len(frame)),
    ]
)
frame["crowding_train"] = crowding[:, 0]
frame["crowding_sm"] = crowding[:, 1]
frame["crowding_car"] = crowding[:, 2]

sd_time, sd_crowd, correlation = 0.10, 0.45, 0.00
covariance = np.array(
    [
        [sd_time**2, correlation * sd_time * sd_crowd],
        [correlation * sd_time * sd_crowd, sd_crowd**2],
    ]
)
person_codes, person_inverse = np.unique(
    frame["person_id"].to_numpy(),
    return_inverse=True,
)
person_wtp = rng.multivariate_normal(
    mean=[0.22, 0.60],
    cov=covariance,
    size=len(person_codes),
)
random_wtp = person_wtp[person_inverse]
alpha = -0.07
utility = np.array([0.25, 0.0, 0.15]) + alpha * (
    cost
    + random_wtp[:, [0]] * time
    + random_wtp[:, [1]] * crowding
)
masked = np.where(available, utility, -np.inf)
probabilities = np.exp(masked - np.max(masked, axis=1, keepdims=True))
probabilities /= probabilities.sum(axis=1, keepdims=True)
frame["choice"] = [
    rng.choice(alternatives, p=row) for row in probabilities
]

data = ChoiceDataset.from_wide(
    frame,
    alternatives=alternatives.tolist(),
    choice="choice",
    variables={
        "time": {"TRAIN": "time_train", "SM": "time_sm", "CAR": "time_car"},
        "cost": {"TRAIN": "cost_train", "SM": "cost_sm", "CAR": "cost_car"},
        "crowding": {
            "TRAIN": "crowding_train",
            "SM": "crowding_sm",
            "CAR": "crowding_car",
        },
    },
    availability={
        "TRAIN": "avail_train",
        "SM": "avail_sm",
        "CAR": "avail_car",
    },
    individual_id="person_id",
)
spec = UtilitySpec()
spec.utility("TRAIN", Beta("ASC_TRAIN", init=0.0))
spec.utility("SM", Beta("ASC_SM", init=0.0, fixed=True))
spec.utility("CAR", Beta("ASC_CAR", init=0.0))


UtilitySpec(utilities=OrderedDict({'TRAIN': Expression(terms=[Term(parameter=Beta(name='ASC_TRAIN', init=0.0, fixed=False), variable=None, multiplier=1.0)]), 'SM': Expression(terms=[Term(parameter=Beta(name='ASC_SM', init=0.0, fixed=True), variable=None, multiplier=1.0)]), 'CAR': Expression(terms=[Term(parameter=Beta(name='ASC_CAR', init=0.0, fixed=False), variable=None, multiplier=1.0)])}))

In [3]:
model = WTPMixedLogit(
    spec,
    cost=Beta("B_COST", init=-0.07),
    cost_variable="cost",
    wtp_coefficients=[
        WTPCoefficient(
            "WTP_TIME",
            "time",
            init=0.18,
            sigma_init=0.10,
            sigma_fixed=True,
        ),
        WTPCoefficient(
            "WTP_CROWDING",
            "crowding",
            init=0.50,
            sigma_init=0.45,
            sigma_fixed=True,
        ),
    ],
    n_draws=48,
    seed=15,
    antithetic=True,
    panel=True,
    correlated=False,
    device=device,
    max_iter=120,
)
result = model.fit(data)
display(HTML(result.report().to_html()))


Model family,WTPMixedLogit
Estimation objective,Maximum simulated log likelihood
TorchDCM version,0.1.0
PyTorch version,2.12.1+cu130
Python version,3.12.13
Operating system,Linux-6.17.0-35-generic-x86_64-with-glibc2.39
Device,cuda
Tensor dtype,float64
Optimizer,torch.optim.LBFGS
Maximum iterations,120
Line search,strong_wolfe
